In [1]:
import pandas as pd
csv_path = "../data/obs_H_M_L.csv"
df = pd.read_csv(csv_path)
df.head()

,id,observed_on,time_observed_at,user_login,license,url,image_url,place_guess,latitude,longitude,positional_accuracy,public_positional_accuracy,taxon_geoprivacy,coordinates_obscured,place_town_name,species_guess,scientific_name,common_name,iconic_taxon_name,taxon_id
0,4276938,2016-10-05,2016-10-05 07:11:15 UTC,mari04,CC-BY-NC,http://www.inaturalist.org/observations/4276938,https://inaturalist-open-data.s3.amazonaws.com...,"Communauté-Urbaine-de-Montréal, CA-QC, CA",45.431838,-73.946628,9,9,NaN,False,NaN,nerprun bourdaine,Frangula alnus,glossy buckthorn,Plantae,55972
1,4439855,2016-10-26,2016-10-26 05:31:07 UTC,mari04,CC-BY-NC,http://www.inaturalist.org/observations/4439855,https://inaturalist-open-data.s3.amazonaws.com...,"Communauté-Urbaine-de-Montréal, CA-QC, CA",45.431513,-73.951803,8,8,NaN,False,NaN,common valerian,Valeriana officinalis,common valerian,Plantae,56160
2,7247406,2017-07-28,2017-07-28 13:59:48 UTC,muddauber,NaN,https://www.inaturalist.org/observations/7247406,https://static.inaturalist.org/photos/9363017/...,"88 Boul Saint-Joseph O, Montréal, QC, CA",45.521108,-73.592236,16,16,NaN,False,NaN,Vigne vierge vraie,Parthenocissus quinquefolia,Virginia creeper,Plantae,50278
3,7518193,2017-08-16,2017-08-16 11:50:47 UTC,whiteoak,CC0,https://www.inaturalist.org/observations/7518193,https://inaturalist-open-data.s3.amazonaws.com...,"2054B Rue Saint-Denis, Montréal, QC, CA",45.516021,-73.566355,10,10,NaN,False,NaN,Virginia creeper,Parthenocissus quinquefolia,Virginia creeper,Plantae,50278
4,7535293,2017-08-17,2017-08-17 15:40:18 UTC,whiteoak,CC0,https://www.inaturalist.org/observations/7535293,https://inaturalist-open-data.s3.amazonaws.com...,"Mount Royal Park, Montréal, QC, CA",45.509002,-73.587749,10,10,NaN,False,NaN,Rosier rugueux,Rosa rugosa,rugosa rose,Plantae,83653


In [29]:
import json, math
from pathlib import Path
import pandas as pd

# ---- Paths (relative to this script's location) ----
CSV_PATH = Path("../data/obs_H_M_L.csv")
OUT_PATH = Path("../data/places.geojson")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)


def clean(v):
    if v is None:
        return ""
    if isinstance(v, float) and math.isnan(v):
        return ""
    return str(v).strip()


def to_float(s):
    s = clean(s)
    if s == "":
        return None
    try:
        f = float(s)
        return None if math.isnan(f) else f
    except ValueError:
        return None


def to_bool(s):
    return clean(s).lower() == "true"


# ---- Risk classification by taxon_id ----
HIGH_RISK_TAXON_IDS = {
    "56061", "169900", "54763", "78157", "55972", "54811",
    "850784", "593040", "914922", "556361", "60467",
    "5903", "1611981",     # Rhamnus cathartica cathartica (both IDs seen in data)
}

MODERATE_RISK_TAXON_IDS = {
    "63337", "130228", "75921", "64540", "82342", "632667",
    "208963", "51741", "1555600", "58727", "55850", "77468",
    "81600", "47892", "116710", "60202", "361265", "64697",
    "129038", "165632", "78707", "204321",
    "487107",              # Petasites japonicus giganteus
}

LOW_RISK_TAXON_IDS = {
    "57278", "124544", "966205", "163682", "60462", "59773",
    "77836", "170851", "55724", "123638", "78237", "77206",
    "163336", "47779", "962637", "47697", "62671", "78589",
    "204339", "59549", "165473", "165474", "79463", "55844",
    "56088", "167957", "78882", "83653", "61321", "72340",
    "122970", "56160", "50278",
    "707386", "462124",    # Lamium galeobdolon subspecies
}


def normalize_taxon_id(v):
    s = clean(v).strip('"').strip("'")
    if s.lower() in ("nan", "none", "null"):
        return ""
    if s.endswith(".0") and s[:-2].isdigit():
        s = s[:-2]
    return s


def risk_for(taxon_id):
    tid = normalize_taxon_id(taxon_id)
    if tid in HIGH_RISK_TAXON_IDS:
        return "high"
    if tid in MODERATE_RISK_TAXON_IDS:
        return "moderate"
    if tid in LOW_RISK_TAXON_IDS:
        return "low"
    return "unknown"


# ---- License handling ----
# We keep every observation regardless of license (facts aren't copyrightable),
# but only show the photo when the license permits reuse.
#
#   CC0, CC-BY, CC-BY-NC   → derivatives allowed → show photo
#   CC-BY-SA, CC-BY-NC-SA  → copyleft on derivatives → drop photo
#   CC-BY-ND, CC-BY-NC-ND  → no derivatives → drop photo
#   blank (All Rights Reserved) → no reuse → drop photo
PHOTO_OK_LICENSES = {"CC0", "CC-BY", "CC-BY-NC"}


def normalize_license(v):
    """Uppercase, collapse whitespace to hyphens, strip junk."""
    s = clean(v).upper()
    s = s.replace(" ", "-").replace("_", "-")
    s = s.replace("--", "-").strip("-")
    return s


# ---- Load ----
df = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
df.columns = [c.strip().lstrip("\ufeff") for c in df.columns]
print(f"Read {len(df)} rows from {CSV_PATH}")

features = []
null_geom = 0
obscured_count = 0
geoprivacy_count = 0
counts = {"high": 0, "moderate": 0, "low": 0, "unknown": 0}
photos_dropped_license = 0
license_seen = {}

for _, row in df.iterrows():
    # --- license: keep observation, maybe drop photo ---
    lic_raw = clean(row.get("license"))
    lic = normalize_license(lic_raw)
    license_seen[lic if lic else "(blank)"] = license_seen.get(lic if lic else "(blank)", 0) + 1

    photo_url = clean(row.get("image_url"))
    if photo_url and lic not in PHOTO_OK_LICENSES:
        photo_url = ""
        photos_dropped_license += 1

    lat = to_float(row.get("latitude"))
    lon = to_float(row.get("longitude"))

    if lat is None or lon is None:
        geometry = None
        null_geom += 1
    else:
        geometry = {"type": "Point", "coordinates": [lon, lat]}

    obscured = to_bool(row.get("coordinates_obscured"))
    if obscured:
        obscured_count += 1

    taxon_geoprivacy = clean(row.get("taxon_geoprivacy"))
    if taxon_geoprivacy:
        geoprivacy_count += 1

    taxon_id = clean(row.get("taxon_id"))
    risk = risk_for(taxon_id)
    counts[risk] += 1

    feature = {
        "type": "Feature",
        "geometry": geometry,
        "properties": {
            "id": clean(row.get("id")),
            "observed_on": clean(row.get("observed_on")),
            "time_observed_at": clean(row.get("time_observed_at")),
            "user_login": clean(row.get("user_login")),
            "license": lic,
            "url": clean(row.get("url")),
            "image_url": photo_url,          # "" when license doesn't permit
            "place_guess": clean(row.get("place_guess")),
            "place_town_name": clean(row.get("place_town_name")),
            "species_guess": clean(row.get("species_guess")),
            "scientific_name": clean(row.get("scientific_name")),
            "common_name": clean(row.get("common_name")),
            "iconic_taxon_name": clean(row.get("iconic_taxon_name")),
            "taxon_id": taxon_id,
            "positional_accuracy": clean(row.get("positional_accuracy")),
            "public_positional_accuracy": clean(row.get("public_positional_accuracy")),
            "coordinates_obscured": obscured,
            "taxon_geoprivacy": taxon_geoprivacy,
            "risk": risk,
        },
    }
    features.append(feature)

geojson = {"type": "FeatureCollection", "features": features}

tmp = OUT_PATH.with_suffix(OUT_PATH.suffix + ".tmp")
with open(tmp, "w", encoding="utf-8") as f:
    json.dump(geojson, f, ensure_ascii=False, indent=2)
tmp.replace(OUT_PATH)

print("\nLicenses found in CSV:")
for k, v in sorted(license_seen.items(), key=lambda kv: -kv[1]):
    print(f"  {k:>12}: {v}")

print(
    f"\nWrote {len(features)} features "
    f"(high={counts['high']}, moderate={counts['moderate']}, "
    f"low={counts['low']}, unknown={counts['unknown']}; "
    f"photos dropped for license: {photos_dropped_license}; "
    f"null_geom={null_geom}, "
    f"{obscured_count} coordinate-obscured, "
    f"{geoprivacy_count} with taxon_geoprivacy) → {OUT_PATH}"
)

Read 8062 rows from ..\data\obs_H_M_L.csv

Licenses found in CSV:
      CC-BY-NC: 5570
       (blank): 1361
         CC-BY: 728
           CC0: 174
   CC-BY-NC-ND: 143
      CC-BY-SA: 52
   CC-BY-NC-SA: 33
      CC-BY-ND: 1

Wrote 8062 features (high=3724, moderate=725, low=3613, unknown=0; photos dropped for license: 1590; null_geom=0, 0 coordinate-obscured, 5833 with taxon_geoprivacy) → ..\data\places.geojson


In [23]:
# Each entry: (presence in Québec, banned under REFEE)
# banned=True means: cultivation and sale are prohibited in Québec.
TAXON_INFO = {
    # --- HIGH RISK ---
    "56061":  ("Established",        True),   # Alliaria petiolata
    "169900": ("Established",        True),   # Trapa natans
    "54763":  ("Established",        False),  # Acer platanoides
    "78157":  ("Established",        True),   # Myriophyllum spicatum
    "55972":  ("Established",        True),   # Frangula alnus
    "54811":  ("Established",        True),   # Rhamnus cathartica
    "1611981": ("Established", True),   # Rhamnus cathartica cathartica
    "850784": ("Established",        True),   # Reynoutria ×bohemica
    "593040": ("Established",        True),   # Reynoutria sachalinensis
    "914922": ("Established",        True),   # Reynoutria japonica
    "556361": ("Established",        True),   # Phragmites australis subsp. australis
    "60467":  ("Established",        True),   # Stratiotes aloides

    # --- MODERATE RISK ---
    "63337":  ("Established",        False),  # Phalaris arundinacea
    "130228": ("Established",        True),   # Heracleum mantegazzianum
    "75921":  ("Absent",             True),   # Cabomba caroliniana
    "64540":  ("Established",        True),   # Celastrus orbiculatus
    "82342":  ("Absent",             True),   # Lonicera maackii
    "632667": ("Established",        True),   # Vincetoxicum rossicum
    "208963": ("Established",        True),   # Vincetoxicum nigrum
    "51741":  ("Established",        True),   # Aegopodium podagraria
    "1555600":("Absent",             True),   # Egeria densa
    "58727":  ("Established",        True),   # Berberis thunbergii
    "55850":  ("Established",        False),  # Petasites hybridus
    "77468":  ("Absent",             True),   # Hydrilla verticillata
    "81600":  ("Established",        True),   # Hydrocharis morsus-ranae
    "47892":  ("Established",        True),   # Impatiens glandulifera
    "116710": ("Absent",             True),   # Microstegium vimineum
    "60202":  ("Absent",             True),   # Myriophyllum aquaticum
    "361265": ("Established",        False),  # Nitellopsis obtusa
    "64697":  ("Absent",             True),   # Elaeagnus umbellata
    "129038": ("Established",        False),  # Petasites japonicus
    "165632": ("Established",        True),   # Najas minor
    "78707":  ("Established",        True),   # Potamogeton crispus
    "204321": ("Established",        True),   # Ficaria verna

    # --- LOW RISK ---
    "57278":  ("Established",        False),  # Ailanthus altissima
    "124544": ("Established",        False),  # Anthriscus sylvestris
    "966205": ("Established",        False),  # Alnus glutinosa
    "163682": ("Established",        True),   # Heracleum sphondylium
    "60462":  ("Established",        False),  # Butomus umbellatus
    "59773":  ("Established",        False),  # Lonicera morrowii
    "77836":  ("Established",        False),  # Lonicera tatarica
    "170851": ("Established",        False),  # Lonicera ×bella
    "55724":  ("Established",        False),  # Symphytum officinale
    "123638": ("Established",        False),  # Acer ginnala
    "78237":  ("Established",        True),   # Nymphoides peltata
    "77206":  ("Established",        False),  # Galium mollugo
    "163336": ("Established",        False),  # Glyceria maxima
    "47779":  ("Established",        False),  # Iris pseudacorus
    "962637": ("Single observation", False),  # Pontederia crassipes
    "47697":  ("Established",        False),  # Hesperis matronalis
    "62671":  ("Absent",             True),   # Pueraria montana
    "78589":  ("Single observation", False),  # Pistia stratiotes
    "204339": ("Established",        False),  # Lamium galeobdolon
    "59549":  ("Established",        False),  # Lupinus polyphyllus
    "165473": ("Established",        False),  # Miscanthus sacchariflorus
    "165474": ("Established",        False),  # Miscanthus sinensis
    "79463":  ("Established",        False),  # Ulmus pumila
    "55844":  ("Established",        False),  # Vinca minor
    "56088":  ("Established",        False),  # Robinia pseudoacacia
    "167957": ("Established",        False),  # Rorippa amphibia
    "78882":  ("Established",        False),  # Rosa multiflora
    "83653":  ("Established",        False),  # Rosa rugosa
    "61321":  ("Established",        False),  # Lythrum salicaria
    "72340":  ("Single observation", False),  # Salvinia spp.
    "122970": ("Established",        False),  # Helianthus tuberosus
    "56160":  ("Established",        False),  # Valeriana officinalis
    "50278":  ("Established",        False),  # Parthenocissus quinquefolia

    "5903":   ("Established",        True),   # Rhamnus cathartica cathartica
    "487107": ("Established",        False),  # Petasites japonicus giganteus
    "707386": ("Established",        False),  # Lamium galeobdolon argentatum
    "462124": ("Established",        False),  # Lamium galeobdolon flavidum
}

In [24]:
# sanity check
import pandas as pd

df = pd.read_csv("../data/obs_H_M_L.csv", dtype=str, keep_default_na=False)

HIGH = {"56061","169900","54763","78157","55972","54811",
        "850784","593040","914922","556361","60467",
        "5903", "1611981"}                              # Rhamnus cathartica cathartica

MOD  = {"63337","130228","75921","64540","82342","632667",
        "208963","51741","1555600","58727","55850","77468",
        "81600","47892","116710","60202","361265","64697",
        "129038","165632","78707","204321",
        "487107"}                            # Petasites japonicus giganteus

LOW  = {"57278","124544","966205","163682","60462","59773",
        "77836","170851","55724","123638","78237","77206",
        "163336","47779","962637","47697","62671","78589",
        "204339","59549","165473","165474","79463","55844",
        "56088","167957","78882","83653","61321","72340",
        "122970","56160","50278",
        "707386","462124"}                   # Lamium galeobdolon subspecies

ids = df["taxon_id"].str.strip()
print("high in csv:    ", ids.isin(HIGH).sum())
print("moderate in csv:", ids.isin(MOD).sum())
print("low in csv:     ", ids.isin(LOW).sum())
print("unknown in csv: ", (~ids.isin(HIGH | MOD | LOW)).sum())
print("total rows:     ", len(df))

high in csv:     3724
moderate in csv: 725
low in csv:      3613
unknown in csv:  0
total rows:      8062


In [25]:
# Checking the unknowns
import pandas as pd

df = pd.read_csv("../data/obs_H_M_L.csv", dtype=str, keep_default_na=False)

HIGH = {"56061","169900","54763","78157","55972","54811",
        "850784","593040","914922","556361","60467","5903","1611981"}
MOD  = {"63337","130228","75921","64540","82342","632667",
        "208963","51741","1555600","58727","55850","77468",
        "81600","47892","116710","60202","361265","64697",
        "129038","165632","78707","204321","487107"}
LOW  = {"57278","124544","966205","163682","60462","59773",
        "77836","170851","55724","123638","78237","77206",
        "163336","47779","962637","47697","62671","78589",
        "204339","59549","165473","165474","79463","55844",
        "56088","167957","78882","83653","61321","72340",
        "122970","56160","50278","707386","462124"}

ids = df["taxon_id"].str.strip()
unknown = df.loc[~ids.isin(HIGH | MOD | LOW)]

print(f"{len(unknown)} unknown rows:\n")
for _, r in unknown.iterrows():
    print(f"  taxon_id={r['taxon_id']!r}  "
          f"scientific_name={r['scientific_name']!r}  "
          f"common_name={r['common_name']!r}  "
          f"species_guess={r['species_guess']!r}")

0 unknown rows:



In [26]:
HIGH_RISK_TAXON_IDS = {
    "56061", "169900", "54763", "78157", "55972", "54811",
    "850784", "593040", "914922", "556361", "60467",
    "5903",          # Rhamnus cathartica cathartica (subsp. of R. cathartica)
}

MODERATE_RISK_TAXON_IDS = {
    "63337", "130228", "75921", "64540", "82342", "632667",
    "208963", "51741", "1555600", "58727", "55850", "77468",
    "81600", "47892", "116710", "60202", "361265", "64697",
    "129038", "165632", "78707", "204321",
    "487107",        # Petasites japonicus giganteus (subsp. of P. japonicus)
}

LOW_RISK_TAXON_IDS = {
    "57278", "124544", "966205", "163682", "60462", "59773",
    "77836", "170851", "55724", "123638", "78237", "77206",
    "163336", "47779", "962637", "47697", "62671", "78589",
    "204339", "59549", "165473", "165474", "79463", "55844",
    "56088", "167957", "78882", "83653", "61321", "72340",
    "122970", "56160", "50278",
    "707386",        # Lamium galeobdolon argentatum (var. of L. galeobdolon)
    "462124",        # Lamium galeobdolon flavidum  (var. of L. galeobdolon)
}

In [27]:
def risk_for(taxon_id):
    tid = normalize_taxon_id(taxon_id)
    if tid in HIGH_RISK_TAXON_IDS:     return "high"
    if tid in MODERATE_RISK_TAXON_IDS: return "moderate"
    if tid in LOW_RISK_TAXON_IDS:      return "low"
    return "unknown"

## 5) Next: Connect to Leaflet
1. In VS Code, right-click `site/index.html` and choose Open with Live Server.
2. `site/app.js` fetches `../data/places.geojson` and adds it to the map with `L.geoJSON(...)`.

In [34]:
import geopandas as gpd
print(gpd.__version__)

1.2.0


In [33]:
!pip install geopandas

   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
   ----------- ---------------------------- 7.1/23.8 MB 35.6 MB/s eta 0:00:01
   ----------------------- ---------------- 14.2/23.8 MB 34.3 MB/s eta 0:00:01
   ---------------------------------- ----- 20.7/23.8 MB 33.5 MB/s eta 0:00:01
   ---------------------------------------- 23.8/23.8 MB 30.0 MB/s  0:00:00
   ---------------------------------------- 0.0/6.9 MB ? eta -:--:--
   ---------------------------------------  6.8/6.9 MB 37.4 MB/s eta 0:00:01
   ---------------------------------------- 6.9/6.9 MB 30.2 MB/s  0:00:00
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 23.6 MB/s  0:00:00

   ---------------------------------------- 0/5 [shapely]
   ---------------------------------------- 0/5 [shapely]
   ---------------------------------------- 0/5 [shapely]
   ---------------------------------------- 0/5 [shapely]
   -------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
import geopandas as gpd

SRC = "../data/especes_exo_envahissantes.GEOJSON"
OUT = "../data/sentinel_wgs84.geojson"     # same folder as places.geojson

gdf = gpd.read_file(SRC)
print("CRS as declared:", gdf.crs)
print("First geometry:", gdf.geometry.iloc[0])

CRS as declared: EPSG:32198
First geometry: POINT (-53788.1462999925 508460.8898010189)


c:\Users\Lukas\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\geopandas\io\file.py:582: UserWarning: Could not parse column 'Code_region_administrative' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


In [37]:
# Declare the source CRS (the coordinates are in EPSG:32198)
gdf = gdf.set_crs(epsg=32198, allow_override=True)

# Reproject to WGS84 (lat/lon) for Leaflet
gdf = gdf.to_crs(epsg=4326)

# Verify: first geometry should now be ~ POINT (-73.x 45.x)
print("CRS after:", gdf.crs)
print("First geometry after:", gdf.geometry.iloc[0])

# Keep only points inside the map's maxBounds
gdf_mtl = gdf.cx[-74.10:-73.30, 45.30:45.75]
print(f"Points inside Montréal bbox: {len(gdf_mtl)} of {len(gdf)}")

# Write out
gdf_mtl.to_file(OUT, driver="GeoJSON")
print(f"Wrote {len(gdf_mtl)} features → {OUT}")

CRS after: EPSG:4326
First geometry after: POINT (-69.23207699999605 48.573328000090406)
Points inside Montréal bbox: 957 of 11015
Wrote 957 features → ../data/sentinel_wgs84.geojson


In [38]:
print(gdf["Nom_region_administrative"].value_counts())

Nom_region_administrative
Capitale-Nationale               2053
Estrie                           1290
Mauricie                         1274
Montérégie                       1153
Laurentides                       910
Chaudière-Appalaches              748
Centre-du-Québec                  584
Bas-Saint-Laurent                 579
Lanaudière                        487
Gaspésie–Îles-de-la-Madeleine     477
Saguenay–Lac-Saint-Jean           400
Outaouais                         374
Côte-Nord                         238
Montréal                          225
Abitibi-Témiscamingue             160
Laval                              30
Nord-du-Québec                     22
Hors Québec                        11
Name: count, dtype: int64


In [40]:
import geopandas as gpd
from pathlib import Path

SRC = Path("../data/especes_exo_envahissantes.GEOJSON")
OUT = Path("../data/sentinel_wgs84.geojson")

gdf = gpd.read_file(SRC)
gdf = gdf.set_crs(epsg=32198, allow_override=True).to_crs(epsg=4326)

gdf_mtl = gdf[gdf["Nom_region_administrative"] == "Montréal"]
print(f"Kept {len(gdf_mtl)} of {len(gdf)} records")

gdf_mtl.to_file(OUT, driver="GeoJSON")
print(f"Wrote → {OUT.resolve()}")

Kept 225 of 11015 records
Wrote → C:\Users\Lukas\Documents\GEOG464\observations-793731.csv\data\sentinel_wgs84.geojson


c:\Users\Lukas\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\geopandas\io\file.py:582: UserWarning: Could not parse column 'Code_region_administrative' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


In [41]:
import geopandas as gpd
chk = gpd.read_file("../data/sentinel_wgs84.geojson")
print("Rows:", len(chk))
print("Bounds:", chk.total_bounds)   # expect ~[-73.98, 45.40, -73.50, 45.70]
print(chk["Nom_region_administrative"].unique())   # should be just ['Montréal']

Rows: 225
Bounds: [-73.942162  45.406114 -73.477979  45.701209]
<StringArray>
['Montréal']
Length: 1, dtype: str


In [42]:
import json, re
from pathlib import Path

INAT = Path("../data/places.geojson")
SENT = Path("../data/sentinel_wgs84.geojson")

# ---- 1. Build name -> risk lookup from the iNat features ----
with open(INAT, encoding="utf-8") as f:
    inat = json.load(f)

def norm_latin(s):
    s = (s or "").strip().lower()
    # strip variety / subspecies / forma rank, keep base binomial + hybrid sign
    s = re.sub(r"\s+(var\.|subsp\.|f\.|forma)\s+\w+", "", s)
    return s

risk_by_latin = {}
risk_by_common = {}
for f in inat["features"]:
    p = f["properties"]
    r = p.get("risk") or "unknown"
    latin  = norm_latin(p.get("scientific_name"))
    common = (p.get("common_name") or "").strip().lower()
    if latin:  risk_by_latin.setdefault(latin, r)
    if common: risk_by_common.setdefault(common, r)

# ---- 2. Assign risk to each Sentinelle feature ----
with open(SENT, encoding="utf-8") as f:
    sent = json.load(f)

matched = 0
unmatched_codes = {}

for feat in sent["features"]:
    p = feat["properties"]
    latin  = norm_latin(p.get("Nom_espece_latin"))
    common = (p.get("Nom_espece_anglais") or "").strip().lower()

    risk = risk_by_latin.get(latin) or risk_by_common.get(common) or "unknown"
    if risk != "unknown":
        matched += 1
    else:
        code = p.get("Code_espece", "?")
        unmatched_codes[code] = unmatched_codes.get(code, 0) + 1

    p["risk"] = risk
    # also expose English name for popups / species filter
    p["common_name_en"] = (p.get("Nom_espece_anglais") or "").strip()

print(f"Risk matched: {matched} of {len(sent['features'])}")
print("Unmatched species codes:")
for code, n in sorted(unmatched_codes.items(), key=lambda kv: -kv[1]):
    print(f"  {code}: {n}")

with open(SENT, "w", encoding="utf-8") as f:
    json.dump(sent, f, ensure_ascii=False, indent=2)

Risk matched: 203 of 225
Unmatched species codes:
  ROSEA: 16
  PANAI: 2
  SCARJ: 2
  FAUXN: 1
  TANCH: 1


In [43]:
# Also translate French common name -> English when iNat has an English name
common_by_latin = {}
for f in inat["features"]:
    p = f["properties"]
    latin = norm_latin(p.get("scientific_name"))
    common = (p.get("common_name") or "").strip()
    if latin and common:
        common_by_latin.setdefault(latin, common)

for feat in sent["features"]:
    p = feat["properties"]
    latin = norm_latin(p.get("Nom_espece_latin"))
    if not p.get("common_name_en") and latin in common_by_latin:
        p["common_name_en"] = common_by_latin[latin]

In [44]:
for feat in sent["features"]:
    feat["properties"]["scientific_name"] = (feat["properties"].get("Nom_espece_latin") or "").strip()

In [45]:
import json, re
from pathlib import Path

INAT = Path("../data/places.geojson")
SENT = Path("../data/sentinel_wgs84.geojson")

# ---- 1. Build name -> risk lookup from the iNat features ----
with open(INAT, encoding="utf-8") as f:
    inat = json.load(f)

def norm_latin(s):
    s = (s or "").strip().lower()
    # strip variety / subspecies / forma rank, keep base binomial + hybrid sign
    s = re.sub(r"\s+(var\.|subsp\.|f\.|forma)\s+\w+", "", s)
    return s

risk_by_latin = {}
risk_by_common = {}
common_by_latin = {}

for f in inat["features"]:
    p = f["properties"]
    r = p.get("risk") or "unknown"
    latin  = norm_latin(p.get("scientific_name"))
    common = (p.get("common_name") or "").strip()
    if latin:
        risk_by_latin.setdefault(latin, r)
        if common:
            common_by_latin.setdefault(latin, common)
    if common:
        risk_by_common.setdefault(common.lower(), r)

# ---- 2. Assign risk + names to each Sentinelle feature ----
with open(SENT, encoding="utf-8") as f:
    sent = json.load(f)

matched = 0
unmatched_codes = {}

for feat in sent["features"]:
    p = feat["properties"]
    latin_raw = (p.get("Nom_espece_latin") or "").strip()
    latin  = norm_latin(latin_raw)
    common_en = (p.get("Nom_espece_anglais") or "").strip()
    common_fr = (p.get("Nom_espece_français") or "").strip()

    # Try risk: latin first, then English common, then French common (rare match)
    risk = (
        risk_by_latin.get(latin)
        or (risk_by_common.get(common_en.lower()) if common_en else None)
        or (risk_by_common.get(common_fr.lower()) if common_fr else None)
        or "unknown"
    )

    if risk != "unknown":
        matched += 1
    else:
        code = p.get("Code_espece", "?")
        unmatched_codes[code] = unmatched_codes.get(code, 0) + 1

    p["risk"] = risk
    p["scientific_name"] = latin_raw
    # Prefer Sentinelle's own English name; fall back to iNat's common name for that Latin
    if not common_en and latin in common_by_latin:
        common_en = common_by_latin[latin]
    p["common_name_en"] = common_en

print(f"Risk matched: {matched} of {len(sent['features'])}")
print("Unmatched species codes:")
for code, n in sorted(unmatched_codes.items(), key=lambda kv: -kv[1]):
    print(f"  {code}: {n}")

with open(SENT, "w", encoding="utf-8") as f:
    json.dump(sent, f, ensure_ascii=False, indent=2)

print(f"Wrote {SENT}")

Risk matched: 203 of 225
Unmatched species codes:
  ROSEA: 16
  PANAI: 2
  SCARJ: 2
  FAUXN: 1
  TANCH: 1
Wrote ..\data\sentinel_wgs84.geojson


In [46]:
import json, re
from pathlib import Path

SENT = Path("../data/sentinel_wgs84.geojson")

# ---- 0. Authoritative invasive-plant list, tiered ----
# Keyed by NORMALIZED scientific name (see norm_latin below).
# Derived from your High / Moderate / Low risk lists.
LIST_HIGH = [
    "alliaria petiolata", "trapa natans", "acer platanoides",
    "myriophyllum spicatum", "frangula alnus", "rhamnus cathartica",
    "reynoutria ×bohemica", "reynoutria sachalinensis", "reynoutria japonica",
    "phragmites australis", "stratiotes aloides",
]
LIST_MODERATE = [
    "phalaris arundinacea", "heracleum mantegazzianum", "cabomba caroliniana",
    "celastrus orbiculatus", "lonicera maackii", "vincetoxicum rossicum",
    "vincetoxicum nigrum", "aegopodium podagraria", "egeria densa",
    "berberis thunbergii", "petasites hybridus", "hydrilla verticillata",
    "hydrocharis morsus-ranae", "impatiens glandulifera", "microstegium vimineum",
    "myriophyllum aquaticum", "nitellopsis obtusa", "elaeagnus umbellata",
    "petasites japonicus", "najas minor", "potamogeton crispus", "ficaria verna",
]
LIST_LOW = [
    "ailanthus altissima", "anthriscus sylvestris", "alnus glutinosa",
    "heracleum sphondylium", "butomus umbellatus", "lonicera morrowii",
    "lonicera tatarica", "lonicera ×bella", "symphytum officinale",
    "acer ginnala", "nymphoides peltata", "galium mollugo", "glyceria maxima",
    "iris pseudacorus", "pontederia crassipes", "hesperis matronalis",
    "pueraria montana", "pistia stratiotes", "lamium galeobdolon",
    "lupinus polyphyllus", "miscanthus sacchariflorus", "miscanthus sinensis",
    "ulmus pumila", "vinca minor", "robinia pseudoacacia", "rorippa amphibia",
    "rosa multiflora", "rosa rugosa", "lythrum salicaria",
    # "salvinia spp." handled as genus-level, see below
    "helianthus tuberosus", "valeriana officinalis",
    "parthenocissus quinquefolia",
]

RISK_BY_NAME = {}
for n in LIST_HIGH:     RISK_BY_NAME.setdefault(n, "high")
for n in LIST_MODERATE: RISK_BY_NAME.setdefault(n, "moderate")
for n in LIST_LOW:      RISK_BY_NAME.setdefault(n, "low")

# Genus-level entries: any species in this genus counts as that tier.
GENUS_TIER = {"salvinia": "low"}

def norm_latin(s):
    s = (s or "").strip().lower()
    s = s.replace("×", "×").replace(" x ", " × ").replace(" x", " ×")  # normalize hybrid sign
    s = re.sub(r"\s+(var\.|subsp\.|f\.|forma)\s+\S+", "", s)  # strip rank + epithet
    s = re.sub(r"\s+", " ", s).strip()
    return s

def lookup_risk(latin_raw):
    """Return (risk, matched) for a raw Latin name. Genus fallback handled here."""
    latin = norm_latin(latin_raw)
    if not latin:
        return ("unknown", False)
    if latin in RISK_BY_NAME:
        return (RISK_BY_NAME[latin], True)
    # Genus-level fallback (e.g. Salvinia spp.)
    genus = latin.split()[0]
    if genus in GENUS_TIER:
        return (GENUS_TIER[genus], True)
    return ("unknown", False)

# ---- 1. Load Sentinelle and filter to invasive plants only ----
with open(SENT, encoding="utf-8") as f:
    sent = json.load(f)

kept, dropped = [], []

for feat in sent["features"]:
    p = feat["properties"]
    latin_raw = (p.get("Nom_espece_latin") or "").strip()
    risk, in_scope = lookup_risk(latin_raw)

    if not in_scope:
        dropped.append(feat)
        continue

    # In scope — enrich the properties.
    p["risk"] = risk
    p["scientific_name"] = latin_raw
    p["common_name_en"] = (p.get("Nom_espece_anglais") or "").strip()
    p["in_scope"] = True
    kept.append(feat)

print(f"Kept {len(kept)} invasive-plant features, dropped {len(dropped)} out-of-scope.")
if dropped:
    from collections import Counter
    dropped_codes = Counter(
        (f["properties"].get("Code_espece", "?"),
         (f["properties"].get("Nom_espece_latin") or "").strip())
        for f in dropped
    )
    print("Dropped (code, latin):")
    for (code, lat), n in dropped_codes.most_common(30):
        print(f"  {code} ({lat}): {n}")

# Any kept features still unknown? (should be zero if list is complete)
still_unknown = [f["properties"].get("Nom_espece_latin")
                 for f in kept if f["properties"]["risk"] == "unknown"]
if still_unknown:
    print("WARNING — kept but risk unknown:", set(still_unknown))

sent["features"] = kept

with open(SENT, "w", encoding="utf-8") as f:
    json.dump(sent, f, ensure_ascii=False, indent=2)

print(f"Wrote {SENT}")

Kept 220 invasive-plant features, dropped 5 out-of-scope.
Dropped (code, latin):
  PANAI (Pastinaca sativa): 2
  SCARJ (Popillia japonica): 2
  TANCH (Tinca tinca): 1
Wrote ..\data\sentinel_wgs84.geojson


In [47]:
import json
from collections import Counter
with open("../data/sentinel_wgs84.geojson", encoding="utf-8") as f:
    s = json.load(f)
print(Counter(f["properties"]["risk"] for f in s["features"]))
print("Total:", len(s["features"]))

Counter({'high': 150, 'low': 37, 'moderate': 33})
Total: 220


In [48]:
!pip install requests


   ---------------------------------------- 0/4 [urllib3]
   ---------------------------------------- 0/4 [urllib3]
   ---------- ----------------------------- 1/4 [idna]
   -------------------- ------------------- 2/4 [charset_normalizer]
   ------------------------------ --------- 3/4 [requests]
   ------------------------------ --------- 3/4 [requests]
   ---------------------------------------- 4/4 [requests]



  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:
import requests
import time
import json

# Paste your list of (scientific name, iNaturalist taxon ID) here.
# You already have these IDs in your "Every species from Risque..." file.
SPECIES_IDS = [
    ("Alliaria petiolata", 56061),
    ("Trapa natans", 169900),
    ("Acer platanoides", 54763),
    ("Myriophyllum spicatum", 78157),
    ("Frangula alnus", 55972),
    ("Rhamnus cathartica", 54811),
    ("Reynoutria × bohemica", 850784),
    ("Reynoutria sachalinensis", 593040),
    ("Reynoutria japonica", 914922),
    ("Phragmites australis subsp. australis", 556361),
    ("Stratiotes aloides", 60467),
    ("Phalaris arundinacea", 63337),
    ("Heracleum mantegazzianum", 130228),
    ("Cabomba caroliniana", 75921),
    ("Celastrus orbiculatus", 64540),
    ("Lonicera maackii", 82342),
    ("Vincetoxicum rossicum", 632667),
    ("Vincetoxicum nigrum", 208963),
    ("Aegopodium podagraria", 51741),
    ("Egeria densa", 1555600),
    ("Berberis thunbergii", 58727),
    ("Petasites hybridus", 55850),
    ("Hydrilla verticillata", 77468),
    ("Hydrocharis morsus-ranae", 81600),
    ("Impatiens glandulifera", 47892),
    ("Microstegium vimineum", 116710),
    ("Myriophyllum aquaticum", 60202),
    ("Nitellopsis obtusa", 361265),
    ("Elaeagnus umbellata", 64697),
    ("Petasites japonicus", 129038),
    ("Najas minor", 165632),
    ("Potamogeton crispus", 78707),
    ("Ficaria verna", 204321),
    ("Ailanthus altissima", 60130), # Note: ID not in your list, placeholder
    # Add the rest of your low-risk and watch species IDs here...
]

def get_summary(taxon_id):
    """Fetch the Wikipedia summary from iNaturalist API."""
    url = f"https://api.inaturalist.org/v1/taxa/{taxon_id}"
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        if data.get("results") and len(data["results"]) > 0:
            summary = data["results"][0].get("wikipedia_summary", "")
            # Clean up: remove HTML tags if any, and extra whitespace
            import re
            summary = re.sub('<[^<]+?>', '', summary).strip()
            return summary
        return ""
    except Exception as e:
        print(f"  Error fetching {taxon_id}: {e}")
        return ""

print("Fetching descriptions from iNaturalist...\n")

for name, tid in SPECIES_IDS:
    print(f"Fetching: {name} (ID: {tid})")
    summary = get_summary(tid)
    
    if summary:
        # Trim to roughly 2-3 sentences (max ~300 chars) for card consistency
        # You can adjust this logic. Simple truncation at a sentence boundary:
        if len(summary) > 300:
            # Try to cut at the last period before 300 chars
            cut = summary[:300]
            last_period = cut.rfind('.')
            if last_period > 150: # Ensure we don't cut too short
                summary = summary[:last_period+1]
            else:
                summary = cut + "..."
        
        # Escape single quotes for JavaScript string
        summary_js = summary.replace("\\", "\\\\").replace("'", "\\'")
        
        print(f"  DESC: '{summary_js}'\n")
    else:
        print(f"  No summary found.\n")
    
    # Be polite to the iNaturalist API (max 60 requests/minute recommended)
    time.sleep(1.1)

print("Done. Copy the DESC lines above into your SPECIES array.")

Fetching descriptions from iNaturalist...

Fetching: Alliaria petiolata (ID: 56061)
  DESC: 'Alliaria petiolata is a biennial flowering plant in the Mustard family, Brassicaceae. It is native to Europe, western and central Asia, and northwestern Africa, from Morocco, Iberia and the British Isles, north to northern Scandinavia, and east to northern Pakistan and western China (Xinjiang).'

Fetching: Trapa natans (ID: 169900)
  DESC: 'The water caltrop is any of three extant species of the genus Trapa: Trapa natans, Trapa bicornis and the endangered Trapa rossica. It is also known as water chestnut, buffalo nut, bat nut, devil pod, ling nut, lin kok, ling kio nut, or singhara. In India its called as singada.'

Fetching: Acer platanoides (ID: 54763)
  DESC: 'Acer platanoides (Norway maple) is a species of maple native to eastern and central Europe and western Asia, from France east to Russia, north to southern Scandinavia and southeast to northern Iran. It is a member of the soapberry and 

In [50]:
SPECIES_IDS = [
    # HIGH RISK (11)
    ("Alliaria petiolata", 56061),
    ("Trapa natans", 169900),
    ("Acer platanoides", 54763),
    ("Myriophyllum spicatum", 78157),
    ("Frangula alnus", 55972),
    ("Rhamnus cathartica", 54811),
    ("Reynoutria x bohemica", 850784),
    ("Reynoutria sachalinensis", 593040),
    ("Reynoutria japonica", 914922),
    ("Phragmites australis", 556361),
    ("Stratiotes aloides", 60467),

    # MODERATE RISK (22)
    ("Phalaris arundinacea", 63337),
    ("Heracleum mantegazzianum", 130228),
    ("Cabomba caroliniana", 75921),
    ("Celastrus orbiculatus", 64540),
    ("Lonicera maackii", 82342),
    ("Vincetoxicum rossicum", 632667),
    ("Vincetoxicum nigrum", 208963),
    ("Aegopodium podagraria", 51741),
    ("Egeria densa", 1555600),
    ("Berberis thunbergii", 58727),
    ("Petasites hybridus", 55850),
    ("Hydrilla verticillata", 77468),
    ("Hydrocharis morsus-ranae", 81600),
    ("Impatiens glandulifera", 47892),
    ("Microstegium vimineum", 116710),
    ("Myriophyllum aquaticum", 60202),
    ("Nitellopsis obtusa", 361265),
    ("Elaeagnus umbellata", 64697),
    ("Petasites japonicus", 129038),
    ("Najas minor", 165632),
    ("Potamogeton crispus", 78707),
    ("Ficaria verna", 204321),

    # LOW RISK (33)
    ("Ailanthus altissima", 57278),
    ("Anthriscus sylvestris", 124544),
    ("Alnus glutinosa", 966205),
    ("Heracleum sphondylium", 163682),
    ("Butomus umbellatus", 60462),
    ("Lonicera morrowii", 59773),
    ("Lonicera tatarica", 77836),
    ("Lonicera x bella", 170851),
    ("Symphytum officinale", 55724),
    ("Acer ginnala", 123638),
    ("Nymphoides peltata", 78237),
    ("Galium mollugo", 77206),
    ("Glyceria maxima", 163336),
    ("Iris pseudacorus", 47779),
    ("Pontederia crassipes", 962637),
    ("Hesperis matronalis", 47697),
    ("Pueraria montana", 62671),
    ("Pistia stratiotes", 78589),
    ("Lamium galeobdolon", 204339),
    ("Lupinus polyphyllus", 59549),
    ("Miscanthus sacchariflorus", 165473),
    ("Miscanthus sinensis", 165474),
    ("Ulmus pumila", 79463),
    ("Vinca minor", 55844),
    ("Robinia pseudoacacia", 56088),
    ("Rorippa amphibia", 167957),
    ("Rosa multiflora", 78882),
    ("Rosa rugosa", 83653),
    ("Lythrum salicaria", 61321),
    ("Salvinia", 72340),
    ("Helianthus tuberosus", 122970),
    ("Valeriana officinalis", 56160),
    ("Parthenocissus quinquefolia", 50278),
]

In [51]:
import requests, time, re

def get_summary(taxon_id):
    url = f"https://api.inaturalist.org/v1/taxa/{taxon_id}"
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        if data.get("results"):
            s = data["results"][0].get("wikipedia_summary", "")
            return re.sub('<[^<]+?>', '', s).strip()
        return ""
    except Exception as e:
        print(f"  Error {taxon_id}: {e}")
        return ""

out = []
for name, tid in SPECIES_IDS:
    print(f"Fetching: {name}")
    summary = get_summary(tid)
    if summary:
        if len(summary) > 300:
            cut = summary[:300]
            lp = cut.rfind('.')
            summary = summary[:lp+1] if lp > 150 else cut + "..."
        js = summary.replace("\\", "\\\\").replace("'", "\\'")
        out.append(f"// {name}\ndesc: '{js}',\n")
    else:
        out.append(f"// {name}\ndesc: '',  // no summary\n")
    time.sleep(1.1)

with open("descriptions.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(out))

print(f"\nDone. Wrote descriptions.txt ({len(out)} entries).")

Fetching: Alliaria petiolata
Fetching: Trapa natans
Fetching: Acer platanoides
Fetching: Myriophyllum spicatum
Fetching: Frangula alnus
Fetching: Rhamnus cathartica
Fetching: Reynoutria x bohemica
Fetching: Reynoutria sachalinensis
Fetching: Reynoutria japonica
Fetching: Phragmites australis
Fetching: Stratiotes aloides
Fetching: Phalaris arundinacea
Fetching: Heracleum mantegazzianum
Fetching: Cabomba caroliniana
Fetching: Celastrus orbiculatus
Fetching: Lonicera maackii
Fetching: Vincetoxicum rossicum
Fetching: Vincetoxicum nigrum
Fetching: Aegopodium podagraria
Fetching: Egeria densa
Fetching: Berberis thunbergii
Fetching: Petasites hybridus
Fetching: Hydrilla verticillata
Fetching: Hydrocharis morsus-ranae
Fetching: Impatiens glandulifera
Fetching: Microstegium vimineum
Fetching: Myriophyllum aquaticum
Fetching: Nitellopsis obtusa
Fetching: Elaeagnus umbellata
Fetching: Petasites japonicus
Fetching: Najas minor
Fetching: Potamogeton crispus
Fetching: Ficaria verna
Fetching: Ailanth